# GDRN D3 — Ứng dụng, PostGIS và k6 trên Kaggle CPU

Tác vụ CPU riêng tư. Java 21, PostgreSQL/PostGIS và k6 chạy trực tiếp trong cùng phiên. Dùng Dataset môi trường ngoại tuyến riêng tư khi DNS Kaggle không khả dụng. Không cần ứng dụng bên ngoài, thông tin xác thực cơ sở dữ liệu có sẵn, đường hầm mạng hay cổng lắng nghe công khai. Chuẩn bị bằng `benchmark/kaggle_api.py`.


## Tác vụ có thể tái lập

Công cụ chuẩn bị API chỉ thay ô cấu hình có nhãn. Ghi lại commit ứng dụng, SHA-256 bộ công cụ đo và gói mã nguồn. Cấu hình `smoke` chạy E07 ghi, E08 đọc trang và đọc bán kính; `ladder` bổ sung đọc trang tại 10/50/100 người dùng ảo. Mỗi phép đo đặt lại cơ sở dữ liệu riêng của tác vụ; phép đọc tạo mẫu qua E07.


In [ ]:
import json
JOB = {"configuration": {"profile": "smoke"}, "bundle": "", "bundleSha256": "", "appCommit": ""}


## Thực thi môi trường cô lập

Chế độ trực tuyến cài gói và biên dịch trên Kaggle. Chế độ ngoại tuyến dùng JAR biên dịch cục bộ cùng tệp thực thi Linux đã kiểm tra từ Dataset riêng tư; ứng dụng, cơ sở dữ liệu và bộ phát tải vẫn chạy trong Kaggle CPU. Tác vụ ngoại tuyến tắt Internet. Bí mật được tạo riêng cho từng tác vụ. Đóng gói ứng dụng không thay thế kiểm thử CI.


In [ ]:
import base64
import hashlib
import io
import json
import shutil
import sys
import tarfile
import tempfile
from pathlib import Path

def unpack(stream, destination):
    with tarfile.open(fileobj=stream, mode="r:gz") as archive:
        for member in archive:
            path = destination / member.name
            if not member.isfile() or not path.resolve().is_relative_to(destination.resolve()):
                raise RuntimeError("Invalid source/runtime archive member")
            path.parent.mkdir(parents=True, exist_ok=True)
            with archive.extractfile(member) as input_file, path.open("wb") as output_file:
                shutil.copyfileobj(input_file, output_file)
            path.chmod(member.mode & 0o777)

def checksum(path):
    digest=hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024*1024), b""): digest.update(block)
    return digest.hexdigest()

if not JOB.get("bundle"):
    raise RuntimeError("Prepare this notebook with benchmark/kaggle_api.py before uploading.")
payload=base64.b64decode(JOB["bundle"], validate=True)
if hashlib.sha256(payload).hexdigest()!=JOB["bundleSha256"]:
    raise RuntimeError("Source bundle checksum mismatch")
source_context=tempfile.TemporaryDirectory(prefix="gdrn-source-",dir="/tmp")
source=Path(source_context.name)
runtime_context=None
offline_runtime=None
try:
    unpack(io.BytesIO(payload),source)
    if JOB.get("runtimeDataset"):
        candidates=[]
        for path in Path("/kaggle/input").rglob("runtime-manifest.json"):
            metadata=json.loads(path.read_text())
            if metadata.get("kind")=="gdrn-offline-runtime" and metadata.get("archiveSha256")==JOB["runtimeArchiveSha256"]:
                candidates.append((path,metadata))
        if len(candidates)!=1: raise RuntimeError("Attach exactly one matching private offline runtime Dataset")
        manifest_path,metadata=candidates[0]
        if metadata["appCommit"]!=JOB["appCommit"]: raise RuntimeError("Offline JAR/source commit mismatch")
        runtime_context=tempfile.TemporaryDirectory(prefix="gdrn-offline-",dir="/tmp")
        runtime_root=Path(runtime_context.name)
        runtime_root.chmod(0o755)
        archive_path=manifest_path.parent/metadata["archive"]
        if archive_path.exists():
            if checksum(archive_path)!=JOB["runtimeArchiveSha256"]: raise RuntimeError("Offline archive checksum mismatch")
            with archive_path.open("rb") as handle: unpack(handle,runtime_root)
        else:
            # Kaggle may expand uploaded archives while ingesting a Dataset.
            roots=[manifest_path.parent/"runtime", manifest_path.parent/metadata["archive"].removesuffix(".tar.gz")/"runtime"]
            roots=[path for path in roots if path.is_dir()]
            if len(roots)!=1: raise RuntimeError("Cannot locate extracted runtime in Dataset")
            shutil.copytree(roots[0],runtime_root/"runtime")
            for path in (runtime_root/"runtime").rglob("*"):
                if path.is_file(): path.chmod(0o755)
        offline_runtime=runtime_root/"runtime"
        for name,digest in metadata["files"].items():
            path=offline_runtime/name
            if not path.resolve().is_relative_to(offline_runtime.resolve()) or checksum(path)!=digest:
                raise RuntimeError("Offline runtime file checksum mismatch")
    sys.path.insert(0,str(source/"benchmark"))
    from kaggle_server import NativeBenchmark
    output=Path("/kaggle/working/d3-results")
    benchmark=NativeBenchmark(source,output,JOB["configuration"],JOB["appCommit"],offline_runtime=offline_runtime)
    benchmark.manifest.update({key:JOB[key] for key in ("runtimeSha256","bundleSha256","runtimeDataset","runtimeArchiveSha256") if key in JOB})
    result=benchmark.execute()
finally:
    source_context.cleanup()
    if runtime_context: runtime_context.cleanup()


## Bằng chứng và giới hạn

Chỉ lưu bằng chứng đo dưới `/kaggle/working/d3-results`: bản kê, hồ sơ tạo mẫu, mẫu k6 thô, tổng kết đã loại thông tin nhạy cảm và CPU/RSS theo nhóm tiến trình. Không xuất tệp cơ sở dữ liệu hoặc thông tin xác thực. Bộ phát tải cạnh tranh CPU với máy chủ; tổng RSS có thể gồm trang dùng chung. Bằng chứng tải về phải liên kết đúng phiên bản notebook Kaggle. Kiểm tra môi trường thành công hoặc notebook báo thành công nhưng chưa hoàn tất phép đo không chứng minh D3 hoàn tất.


In [ ]:
print(json.dumps({"status": result["status"], "appCommit": result["appCommit"], "runs": len(result["runs"]), "cleanup": result["cleanup"]["status"], "output": str(output)}, indent=2))
